# Scenario Playbook: Customer Solutions with Claude

**Goal**: Practice building production-quality prompt-based solutions for realistic customer problems.
Each scenario walks through the full arc: understanding the customer need, designing the solution,
implementing it, evaluating it, and presenting it back to the customer.

**Interview Context**: Anthropic's Customer Scenarios interview for Applied AI Engineer.
You'll be given a vague customer request and need to design a working solution live.
These 6 scenarios cover the most common patterns you'll encounter.

| # | Pattern | Customer | Key Skill |
|---|---------|----------|-----------|
| 1 | Classification | Nonprofit helpline | Few-shot prompting, confidence calibration |
| 2 | Extraction | Grant foundation | Structured output, XML/JSON formatting |
| 3 | Summarization | Health research org | Audience adaptation, output templating |
| 4 | Content Generation | Fundraising team | Tone control, personalization |
| 5 | Q&A over Documents | Social services org | Grounding, hallucination prevention |
| 6 | Multi-step Workflow | Intake processing org | Prompt chaining, pipeline design |

---

## Setup

In [ ]:
!pip install anthropic pydantic -q

In [ ]:
import anthropic
import json
import textwrap

# Colab: use Secrets (key icon in sidebar)
# from google.colab import userdata
# client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5-20251001"


def call_claude(system_prompt, user_message, model=MODEL, max_tokens=2048):
    """Simple wrapper for the Messages API."""
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text


def print_result(label, text, width=90):
    """Pretty-print a labeled result."""
    border = "=" * width
    print(f"\n{border}")
    print(f"  {label}")
    print(border)
    print(text)
    print()


print("Client ready. Model:", MODEL)

---

# Scenario 1: CLASSIFICATION -- Support Ticket Routing

---

## Customer Brief

**Who they are**: A nonprofit helpline serving low-income communities. They receive 200+ emails per day from people seeking help with various issues.

**What they said** (paraphrased from the intake call):

> "We have three staff members who spend their entire morning reading emails and forwarding them to the right department. It takes about 3 hours every day. Sometimes things get misrouted and people wait days for a response. We have five departments: Legal Aid, Benefits Enrollment, Housing Assistance, Mental Health Services, and General Inquiry. Can AI just... do this for us?"

**Constraints they mentioned**:
- Must be accurate -- misrouting a crisis email to General Inquiry is dangerous
- Some emails mention multiple issues; they want a primary category
- They want to understand WHY the AI chose a category (for auditing)

## Discovery Notes

**Questions I'd ask and what I'd learn:**

1. **"Can you share 20-30 example emails with their correct routing?"** -- This gives us few-shot examples and reveals edge cases. We learn that many emails are messy: typos, multiple languages, emotional tone.

2. **"What happens when an email touches multiple departments?"** -- They say: route to the most *urgent* need first. Mental health crises always take priority. After that, Legal > Benefits > Housing > General.

3. **"What does 'misrouting' look like today? What are the common mistakes?"** -- Benefits vs. Legal is the most confused pair (e.g., "I lost my disability benefits" could be either). Housing vs. General is another (e.g., "I need help" with no specifics).

4. **"How would this integrate into your workflow?"** -- They use a shared Gmail inbox. They'd want the category and confidence level, and a human would review anything below 80% confidence.

5. **"What's the cost of a false positive vs. false negative for each category?"** -- Misrouting a mental health crisis is the highest-cost error. Sending something to General that should be Legal is annoying but not dangerous.

## Solution

In [ ]:
TICKET_ROUTING_SYSTEM = """You are a support ticket routing assistant for a nonprofit helpline serving low-income communities.

<role>
Your job is to read incoming emails and classify each one into exactly ONE primary department for routing. You must also provide a confidence level and brief reasoning.
</role>

<categories>
1. MENTAL_HEALTH -- Any mention of emotional distress, suicidal ideation, substance abuse, counseling needs, trauma, anxiety, depression, or crisis situations. THIS IS THE HIGHEST PRIORITY -- if there is ANY indication of mental health crisis, classify here regardless of other issues mentioned.
2. LEGAL_AID -- Questions about legal rights, eviction notices, court dates, custody disputes, immigration status, discrimination complaints, or requests for legal representation.
3. BENEFITS_ENROLLMENT -- Questions about applying for or maintaining government benefits (SNAP, Medicaid, SSI/SSDI, WIC, TANF, unemployment), eligibility questions, missing benefit payments, or recertification.
4. HOUSING_ASSISTANCE -- Requests for help finding housing, rent assistance programs, utility shutoff prevention, Section 8 vouchers, shelter placement, or landlord disputes that are NOT legal in nature.
5. GENERAL_INQUIRY -- Anything that doesn't clearly fit the above categories: general questions about the org, volunteer inquiries, donation receipts, event information, or vague requests that need human triage.
</categories>

<priority_rules>
- When an email mentions multiple issues, route to the MOST URGENT department.
- Priority order: MENTAL_HEALTH > LEGAL_AID > BENEFITS_ENROLLMENT > HOUSING_ASSISTANCE > GENERAL_INQUIRY
- If you detect ANY sign of crisis or self-harm, ALWAYS classify as MENTAL_HEALTH regardless of the stated request.
</priority_rules>

<examples>
<example>
<email>Hi, I got a letter saying my food stamps are being cut off next month. I don't understand why. I sent in my recertification paperwork two weeks ago. Can someone help me figure out what happened? My case number is 4492817. Thank you, Maria</email>
<classification>
<category>BENEFITS_ENROLLMENT</category>
<confidence>95</confidence>
<reasoning>Clear request about maintaining SNAP benefits (food stamps). Mentions recertification paperwork and case number. No indication of crisis or legal issue -- this is a straightforward benefits administration question.</reasoning>
</classification>
</example>

<example>
<email>I need help ASAP. My landlord put a notice on my door saying I have 3 days to leave. I have two kids and nowhere to go. I can't stop crying and I don't know what to do anymore. I feel like giving up. Please someone help me.</email>
<classification>
<category>MENTAL_HEALTH</category>
<confidence>90</confidence>
<reasoning>Although this email mentions an eviction notice (Legal) and housing instability (Housing), the phrases "can't stop crying," "don't know what to do anymore," and "feel like giving up" are strong indicators of emotional crisis and possible suicidal ideation. Per priority rules, MENTAL_HEALTH takes precedence. The legal and housing issues can be addressed after the crisis is stabilized.</reasoning>
</classification>
</example>

<example>
<email>Hello, my name is James. I got a letter from the court about my custody hearing next Tuesday. I don't have a lawyer and I can't afford one. My ex's attorney keeps sending me documents I don't understand. Is there any way to get free legal help? I'm really worried about losing time with my daughter.</email>
<classification>
<category>LEGAL_AID</category>
<confidence>97</confidence>
<reasoning>Clear legal matter: custody hearing, need for legal representation, court documents. The emotional concern about his daughter is natural parental worry, not a mental health crisis. This is a straightforward legal aid referral.</reasoning>
</classification>
</example>

<example>
<email>Do you guys have any volunteer opportunities? I'm a retired teacher and I'd like to help out in the community. I can come in on Tuesdays and Thursdays. Also, is there parking at your office? Thanks!</email>
<classification>
<category>GENERAL_INQUIRY</category>
<confidence>99</confidence>
<reasoning>Volunteer inquiry with logistical question about parking. Does not relate to any service department -- this is a general organizational inquiry.</reasoning>
</classification>
</example>
</examples>

<output_format>
For each email, respond with ONLY the following XML structure:
<classification>
<category>[EXACTLY one of: MENTAL_HEALTH, LEGAL_AID, BENEFITS_ENROLLMENT, HOUSING_ASSISTANCE, GENERAL_INQUIRY]</category>
<confidence>[0-100 integer]</confidence>
<reasoning>[1-3 sentences explaining why this category was chosen and why others were ruled out]</reasoning>
</classification>
</output_format>"""

# Quick test with one email
test_email = """Subject: Need help with rent

Hi there, I lost my job two months ago and I'm behind on rent. My landlord says if I 
don't pay by the end of the month he'll start eviction proceedings. I applied for 
unemployment but haven't heard back. Is there any emergency rental assistance available? 
I have three kids and I really can't lose this apartment."""

result = call_claude(TICKET_ROUTING_SYSTEM, f"<email>{test_email}</email>")
print_result("Test Classification", result)

## Quick Eval

In [ ]:
routing_test_cases = [
    {
        "label": "Clear Benefits Case",
        "email": "My Medicaid card stopped working at the pharmacy yesterday. They said my coverage was terminated but I never got any notice. I need my heart medication. Case #MED-20241.",
        "expected": "BENEFITS_ENROLLMENT",
    },
    {
        "label": "Ambiguous: Housing or Legal?",
        "email": "My landlord changed the locks while I was at work and put all my stuff on the sidewalk. He said I owe him $2,000 in back rent. I'm staying at my sister's place. What can I do?",
        "expected": "LEGAL_AID",  # Illegal lockout is a legal matter
    },
    {
        "label": "Hidden Mental Health Crisis",
        "email": "I need information about your housing programs. I've been sleeping in my car for a week. Honestly I don't even care anymore. Nothing ever works out. What's the point of even trying. But anyway if you have any shelter info that would be fine I guess.",
        "expected": "MENTAL_HEALTH",  # "don't care anymore" + "what's the point" = crisis signals
    },
    {
        "label": "Multi-issue: Benefits + Housing",
        "email": "Hello, I'm being discharged from a rehabilitation facility next week and I need to find somewhere to live. I also need to reapply for SSI because my benefits were suspended while I was in rehab. Can you help with both? Thank you.",
        "expected": "BENEFITS_ENROLLMENT",  # SSI reactivation is more urgent than housing search
    },
    {
        "label": "Vague / Insufficient Information",
        "email": "help me please. i need help. can someone call me at 555-0147. its urgent.",
        "expected": "GENERAL_INQUIRY",  # Too vague to classify, but "urgent" may warrant human triage
    },
]

print("SCENARIO 1: Support Ticket Routing -- Evaluation")
print("=" * 70)

for i, case in enumerate(routing_test_cases, 1):
    result = call_claude(TICKET_ROUTING_SYSTEM, f"<email>{case['email']}</email>")
    print(f"\n--- Test {i}: {case['label']} ---")
    print(f"Expected: {case['expected']}")
    print(f"Result:\n{result}")
    print()

## Demo Talking Points

**What I'd say when presenting this to the customer:**

1. **"Here's what we built and how it works."** The system reads each incoming email and assigns it to one of your five departments with a confidence score and a written explanation. The explanation is there so your staff can audit any decision they disagree with.

2. **"We built in a safety net."** The system is designed to be *conservative* about mental health -- if there's any signal of crisis, it routes to Mental Health first, even if the email is ostensibly about housing or benefits. You told us that misrouting a crisis is the most dangerous mistake, so we optimized for that.

3. **"You keep a human in the loop."** For anything under 80% confidence, the system flags it for manual review rather than auto-routing. This catches the genuinely ambiguous cases while still handling the clear-cut 70-80% of emails automatically.

4. **"Here are the results on your real data."** *(Show eval results.)* We tested on representative examples including edge cases -- ambiguous emails, multi-issue emails, and crisis signals hidden in mundane requests. The accuracy on our test set is [X]%, and every error we saw was a borderline case that your staff also disagrees on.

5. **"Next steps: integration and monitoring."** We'd connect this to your Gmail inbox via API, run it in shadow mode for two weeks (AI classifies but humans still route), compare agreement rates, then go live with the confidence threshold.

---

# Scenario 2: EXTRACTION -- Grant Application Processing

---

## Customer Brief

**Who they are**: A philanthropic foundation that funds community health and education programs. They review 500 grant applications per funding cycle (twice a year).

**What they said**:

> "Our program officers spend the first two weeks of each cycle just reading applications and entering data into a spreadsheet. We need to extract: organization name, requested amount, project summary, target population, timeline, and budget breakdown. Every application is written differently -- some are very formal, others are basically a long email. Can you automate the data entry part?"

**Constraints**:
- Must handle wildly different writing styles and formats
- Extracted amounts must be accurate (financial data)
- If a field is missing or unclear, they want it flagged rather than guessed

## Discovery Notes

1. **"What does your spreadsheet look like?"** -- They shared the column headers. This tells us the exact output schema we need. Key insight: "budget breakdown" needs to be a list of line items, not a single number.

2. **"What's 'good enough' accuracy for each field?"** -- Org name and amount must be 100% accurate. Project summary can be approximate. Timeline can be approximate. Budget breakdown must capture all line items but exact wording can vary.

3. **"How do you handle incomplete applications?"** -- They want a flag like `MISSING` or `UNCLEAR` rather than the model guessing. This is critical -- better to flag for human review than to insert a wrong number.

4. **"What format should the output be in?"** -- JSON that can be imported into their existing Google Sheets workflow. Each application becomes one row.

## Solution

In [ ]:
GRANT_EXTRACTION_SYSTEM = """You are a data extraction assistant for a philanthropic foundation that reviews grant applications.

<role>
Your job is to read a grant application and extract structured data from it. You must be precise with financial figures and organization names. If any field is missing, ambiguous, or cannot be confidently determined from the text, mark it as "MISSING" or "UNCLEAR" rather than guessing.
</role>

<output_schema>
Return a JSON object with exactly these fields:
{
  "organization_name": "string -- exact legal name of the applying organization",
  "requested_amount": "number -- total dollar amount requested, as an integer (no cents)",
  "project_title": "string -- name or title of the proposed project",
  "project_summary": "string -- 2-3 sentence summary of what the project will do",
  "target_population": "string -- who the project serves (demographics, geography, count if stated)",
  "timeline": {
    "start_date": "string -- planned start date or 'UNCLEAR'",
    "end_date": "string -- planned end date or 'UNCLEAR'",
    "duration_months": "number or 'UNCLEAR'"
  },
  "budget_breakdown": [
    {"item": "string -- what the money is for", "amount": "number -- dollar amount"}
  ],
  "flags": ["list of any fields that were MISSING or UNCLEAR, with explanation"]
}
</output_schema>

<extraction_rules>
- NEVER invent or assume financial figures. If an amount is not explicitly stated, mark it UNCLEAR.
- Organization name: Use the exact name as written. Do not correct spelling or expand abbreviations.
- Budget breakdown: Extract every line item mentioned. If percentages are given instead of dollar amounts, calculate the dollar amounts based on the total requested amount.
- Timeline: Look for explicit dates, months, or durations. "This year" or "soon" = UNCLEAR.
- Target population: Include any demographic details, geographic scope, and estimated number of people served.
- If the application is very short or informal, extract what you can and flag everything else.
</extraction_rules>

<output_format>
Respond with ONLY the JSON object. No preamble, no explanation, no markdown code fences. Just valid JSON.
</output_format>"""

# Test with a well-structured application
sample_application_1 = """GRANT APPLICATION -- Community Health Initiative

Organization: Riverside Community Health Partners, Inc.
Contact: Dr. Sarah Chen, Executive Director

Project Title: "Healthy Families, Healthy Futures" Maternal Health Outreach Program

Funding Request: $175,000

Project Description:
Riverside Community Health Partners seeks funding to expand our maternal health outreach 
program to serve an additional 300 pregnant women and new mothers in the East Riverside 
community, where 40% of residents lack access to prenatal care. The program provides 
free prenatal screenings, nutrition education, and postpartum support through a team of 
community health workers who conduct home visits. We have operated this program 
successfully in West Riverside since 2019, serving over 1,200 women with a 94% healthy 
birth outcome rate.

Target Population: Pregnant women and new mothers (up to 12 months postpartum) in East 
Riverside census tracts 4201-4208. Primarily Latina and Black women aged 18-35, many 
uninsured or on Medicaid. Estimated 300 women served in Year 1.

Timeline: July 2025 through June 2026 (12 months)

Budget:
- Community Health Workers (3 FTE): $105,000
- Prenatal screening supplies and equipment: $25,000
- Nutrition education materials and food boxes: $20,000
- Transportation (home visits): $12,000
- Program evaluation and data collection: $8,000
- Administrative overhead (5%): $5,000

Total: $175,000"""

result = call_claude(GRANT_EXTRACTION_SYSTEM, f"<application>{sample_application_1}</application>")
print_result("Extracted Data (Well-Structured Application)", result)

# Parse and pretty-print to verify valid JSON
try:
    parsed = json.loads(result)
    print("Valid JSON: YES")
    print(f"Org: {parsed['organization_name']}")
    print(f"Amount: ${parsed['requested_amount']:,}")
    print(f"Budget items: {len(parsed['budget_breakdown'])}")
    print(f"Flags: {parsed['flags']}")
except json.JSONDecodeError as e:
    print(f"JSON Parse Error: {e}")

## Quick Eval

In [ ]:
# Test with an informal, messy application
sample_application_2 = """Hi there,

I'm writing on behalf of the Oak Street Youth Center. We've been running afterschool 
programs for kids in the Parkview neighborhood for about 6 years now. We're looking 
for $50,000 to start a new coding and robotics club.

Basically we want to buy 20 laptops and some robot kits (maybe those LEGO Mindstorm 
ones?) and hire a part-time instructor who knows programming. We'd run it twice a 
week after school for middle schoolers -- probably about 40-50 kids would sign up 
based on our waitlist for other programs.

We're hoping to start in the fall, maybe September? And run it through the school 
year. The laptops would be about $15,000, the robot kits maybe $5,000, and the 
instructor would be around $25,000 for the year. We'd use the rest for supplies and 
snacks for the kids.

Let me know if you need anything else!

Thanks,
Marcus Johnson
Program Director"""

# Test with a very incomplete application
sample_application_3 = """Green Valley Food Bank

We are requesting a grant to expand our mobile food pantry program. We currently serve 
12 sites but need to add 5 more in underserved rural areas of Jefferson County. 
Approximately 2,000 additional families would gain access to weekly food distributions.

The expansion requires a new refrigerated truck and additional volunteers. We plan to 
begin operations as soon as funding is secured."""

test_applications = [
    ("Well-Structured (Riverside Health)", sample_application_1),
    ("Informal/Conversational (Oak Street Youth)", sample_application_2),
    ("Incomplete (Green Valley Food Bank)", sample_application_3),
]

print("SCENARIO 2: Grant Extraction -- Evaluation")
print("=" * 70)

for label, app_text in test_applications:
    result = call_claude(GRANT_EXTRACTION_SYSTEM, f"<application>{app_text}</application>")
    try:
        parsed = json.loads(result)
        print(f"\n--- {label} ---")
        print(f"  Org:       {parsed['organization_name']}")
        print(f"  Amount:    ${parsed['requested_amount']:,}" if isinstance(parsed['requested_amount'], (int, float)) else f"  Amount:    {parsed['requested_amount']}")
        print(f"  Project:   {parsed['project_title']}")
        print(f"  Timeline:  {parsed['timeline']['start_date']} to {parsed['timeline']['end_date']} ({parsed['timeline']['duration_months']} months)")
        print(f"  Budget items: {len(parsed['budget_breakdown'])}")
        for item in parsed['budget_breakdown']:
            amt = item['amount']
            print(f"    - {item['item']}: ${amt:,}" if isinstance(amt, (int, float)) else f"    - {item['item']}: {amt}")
        if parsed['flags']:
            print(f"  FLAGS: {parsed['flags']}")
        else:
            print(f"  FLAGS: None")
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"\n--- {label} ---")
        print(f"  Parse error: {e}")
        print(f"  Raw output: {result[:300]}")

## Demo Talking Points

1. **"The system extracts structured data from any writing style."** We tested on formal applications, informal emails, and incomplete submissions. It handles all of them and produces consistent JSON output that maps directly to your spreadsheet columns.

2. **"It never guesses -- it flags."** When the Green Valley application didn't include a dollar amount or specific timeline, the system flagged those fields as MISSING or UNCLEAR instead of making something up. Your program officers only need to review flagged fields, not re-read entire applications.

3. **"Financial accuracy is built in."** We instructed the model to extract amounts exactly as stated and to calculate line items from percentages when needed. The budget breakdown sums are validated against the total requested amount.

4. **"This plugs into your existing workflow."** The JSON output maps 1:1 to your spreadsheet columns. We can set up a batch process that reads applications from your upload folder, extracts the data, and populates the spreadsheet -- your officers just review and approve.

---

# Scenario 3: SUMMARIZATION -- Research Paper Digests

---

## Customer Brief

**Who they are**: A health research organization with a 12-person leadership team (executive director, department heads, board liaison). They track developments in public health, epidemiology, and health policy.

**What they said**:

> "One of our research analysts spends about 2 hours every morning reading new papers and writing a daily digest email for leadership. It's bullet points -- what was published, why it matters, what we should do about it. She's great at it but she's also our most expensive analyst and this isn't the best use of her time. Can Claude do this?"

**Constraints**:
- Audience is executives, not scientists -- no jargon, focus on implications
- Must flag if a finding contradicts something the org has previously communicated
- Must distinguish between strong evidence and preliminary findings

## Discovery Notes

1. **"Can you share some of the digests your analyst has written?"** -- These become our output format template. Key pattern: Title, 3 bullets for key findings, a "So What" section for implications, and a one-line limitations note.

2. **"Who exactly reads this?"** -- The ED skims headlines. Department heads read findings for their area. Board liaison reads everything carefully. This tells us the summary needs to work at multiple levels of engagement.

3. **"How do you define 'strong' vs 'preliminary'?"** -- Randomized controlled trials and large cohort studies = strong. Small sample sizes, observational studies, preprints = preliminary. Meta-analyses = very strong.

4. **"How do papers get selected?"** -- The analyst has RSS feeds and email alerts. She selects 3-5 papers per day. The selection part stays human; we're automating the writing.

## Solution

In [ ]:
RESEARCH_DIGEST_SYSTEM = """You are a research digest writer for a health research organization's leadership team.

<role>
Your job is to read research paper abstracts or excerpts and produce concise, executive-friendly summaries. Your audience is non-scientist leadership: an executive director, department heads, and a board liaison. They need to understand what was found and why it matters -- not HOW the research was conducted.
</role>

<audience_guidelines>
- NO scientific jargon. Replace technical terms with plain language (e.g., "randomized controlled trial" becomes "gold-standard clinical study").
- Focus on IMPLICATIONS, not methods. Leadership cares about "What does this mean for us?" not "They used a Cox proportional hazards model."
- Numbers should be meaningful: "cut hospital readmissions by 30%" is better than "OR 0.70, 95% CI 0.58-0.85."
- Be honest about evidence quality. Distinguish between strong evidence and early-stage findings.
</audience_guidelines>

<evidence_quality_rubric>
- STRONG EVIDENCE: Randomized controlled trials (n>500), large cohort studies (n>10,000), systematic reviews, meta-analyses. Say: "Strong evidence shows..."
- MODERATE EVIDENCE: Cohort studies (n=100-10,000), well-designed observational studies. Say: "Growing evidence suggests..."
- PRELIMINARY EVIDENCE: Small studies (n<100), case series, preprints, animal studies, cross-sectional studies. Say: "Early research hints that..." or "A small study found..."
</evidence_quality_rubric>

<output_format>
For each paper, produce this exact structure:

## [Plain-language title -- rewrite the academic title to be clear and direct]

**Evidence quality**: [STRONG / MODERATE / PRELIMINARY] -- [one-line justification]

**Key findings**:
- [Finding 1 -- most important, in plain language with key numbers]
- [Finding 2 -- supporting detail]
- [Finding 3 -- additional context if relevant]

**So what?**: [2-3 sentences on what this means for the organization, programs, or public health messaging. Be specific and actionable.]

**Limitations**: [One sentence on what would make us MORE confident -- bigger study, longer follow-up, different population, etc.]
</output_format>"""

# Sample abstract -- realistic length and content
sample_abstract = """Title: Community Health Worker-Led Interventions and Cardiovascular Risk Factor Control 
Among Low-Income Adults: A Cluster-Randomized Controlled Trial

Authors: Martinez-Garcia R, Chen W, Okonkwo DC, et al.
Journal: JAMA Internal Medicine, 2025

Abstract:
IMPORTANCE: Cardiovascular disease (CVD) disproportionately affects low-income populations, 
who face barriers to accessing primary care and managing chronic conditions.

OBJECTIVE: To evaluate the effectiveness of a community health worker (CHW)-led intervention 
on cardiovascular risk factor control among low-income adults.

DESIGN, SETTING, AND PARTICIPANTS: Cluster-randomized controlled trial conducted at 24 
federally qualified health centers (FQHCs) across 6 US states from March 2022 to September 
2024. Participants were 2,847 adults aged 35-75 with at least 2 uncontrolled CVD risk 
factors (hypertension, diabetes, hyperlipidemia, or obesity) and household income below 
200% of the federal poverty level.

INTERVENTION: CHWs conducted monthly home visits for 18 months, providing health education, 
medication adherence support, care coordination, and social needs screening with referrals. 
Control sites received enhanced usual care with printed educational materials.

MAIN OUTCOMES AND MEASURES: Primary outcome was the proportion of participants achieving 
control of at least 2 CVD risk factors at 18 months. Secondary outcomes included individual 
risk factor control, emergency department visits, and hospitalization rates.

RESULTS: Among 2,847 participants (mean age 54.3 years; 62% female; 44% Hispanic, 31% 
non-Hispanic Black), the intervention group showed significantly higher rates of achieving 
control of >=2 risk factors compared with control (48.2% vs 29.7%; adjusted risk ratio, 
1.58; 95% CI, 1.34-1.86; P<.001). Blood pressure control improved from 31% to 52% in the 
intervention group vs 30% to 37% in controls. HbA1c control (<8%) improved from 43% to 
61% vs 42% to 48%. ED visits decreased by 23% and hospitalizations by 31% in the 
intervention group. The intervention cost $1,847 per participant over 18 months, with an 
estimated net savings of $3,200 per participant in reduced acute care utilization.

CONCLUSIONS AND RELEVANCE: A CHW-led intervention significantly improved cardiovascular 
risk factor control among low-income adults, reduced acute care utilization, and generated 
net cost savings. These findings support the integration of CHWs into primary care teams 
serving underserved populations."""

result = call_claude(RESEARCH_DIGEST_SYSTEM, f"<paper>{sample_abstract}</paper>")
print_result("Research Digest -- Executive Audience", result)

In [ ]:
# Now show how changing the audience changes the output

TECHNICAL_DIGEST_SYSTEM = """You are a research digest writer for a team of epidemiologists and biostatisticians.

<role>
Your audience is technically sophisticated researchers. They want to know about methodology, 
statistical rigor, and how this study compares to existing literature. Use precise technical 
language. Focus on study design, effect sizes, confidence intervals, and potential confounders.
</role>

<output_format>
## [Original paper title]

**Study design**: [Design type, sample size, duration, setting]

**Key results**:
- [Primary outcome with effect size and CI]
- [Secondary outcomes]
- [Cost-effectiveness data if available]

**Methodological notes**: [Strengths and weaknesses of the study design, potential confounders, generalizability concerns]

**Context**: [How this fits with existing literature, what questions remain]
</output_format>"""

exec_result = call_claude(RESEARCH_DIGEST_SYSTEM, f"<paper>{sample_abstract}</paper>")
tech_result = call_claude(TECHNICAL_DIGEST_SYSTEM, f"<paper>{sample_abstract}</paper>")

print_result("EXECUTIVE Audience Version", exec_result)
print("\n" + "*" * 90 + "\n")
print_result("TECHNICAL Audience Version", tech_result)

## Quick Eval

In [ ]:
# Test with a weak/preliminary study to see how evidence quality is handled
preliminary_abstract = """Title: Exploring the Association Between Urban Green Space Access and Self-Reported 
Mental Well-Being: A Cross-Sectional Survey

Authors: Thompson LK, Patel NR
Journal: Preprint (not yet peer-reviewed), uploaded to medRxiv January 2025

Abstract:
We surveyed 87 residents of a mid-sized Midwestern city about their proximity to parks 
and green spaces and their self-reported mental well-being using the WHO-5 Well-Being 
Index. Participants living within a 10-minute walk of a park reported higher well-being 
scores (mean 68.2 vs 59.4, p=0.03). We did not control for income, education, or 
pre-existing mental health conditions. The survey was conducted online and had a 12% 
response rate. These preliminary findings suggest that urban green space may contribute 
to mental well-being, though larger, controlled studies are needed."""

# Test with a meta-analysis (very strong evidence)
meta_analysis_abstract = """Title: Effectiveness of School-Based Mental Health Programs in Reducing Adolescent 
Depression and Anxiety: A Systematic Review and Meta-Analysis of 47 Randomized Controlled Trials

Authors: Nakamura Y, Williams S, Osei-Mensah A, et al.
Journal: The Lancet Psychiatry, 2025

Abstract:
We conducted a systematic review and meta-analysis of 47 RCTs (N=38,429 students aged 
11-18) evaluating school-based mental health programs. Programs that included cognitive 
behavioral therapy (CBT) components showed significant reductions in depressive symptoms 
(standardized mean difference [SMD] -0.38, 95% CI -0.47 to -0.29; I-squared=42%) and 
anxiety symptoms (SMD -0.31, 95% CI -0.39 to -0.23; I-squared=38%). Programs delivered 
by trained teachers were as effective as those delivered by mental health professionals 
(p for interaction = 0.72). Effects were sustained at 12-month follow-up but diminished 
by 24 months without booster sessions. Universal programs (offered to all students) were 
less effective than targeted programs (offered to at-risk students) for depression 
(SMD -0.28 vs -0.51, p=0.003) but equally effective for anxiety. Cost per student ranged 
from $45-$180 per school year. No serious adverse events were reported in any trial."""

summaries = [
    ("Strong: CHW Cardiovascular Trial", sample_abstract),
    ("Preliminary: Green Space Survey", preliminary_abstract),
    ("Very Strong: School Mental Health Meta-Analysis", meta_analysis_abstract),
]

print("SCENARIO 3: Research Digests -- Evidence Quality Comparison")
print("=" * 70)

for label, abstract in summaries:
    result = call_claude(RESEARCH_DIGEST_SYSTEM, f"<paper>{abstract}</paper>")
    print_result(label, result)

## Demo Talking Points

1. **"The system writes for your audience, not for scientists."** Compare the executive version to the technical version -- same paper, completely different framing. Your leadership team gets plain-language bullets they can act on. No one has to Google what "adjusted risk ratio" means.

2. **"It's honest about evidence quality."** Notice how the preprint study is flagged as PRELIMINARY with a clear explanation of why (small sample, not peer-reviewed, no controls). Your leadership won't accidentally cite a weak study in a board presentation.

3. **"The 'So What' section is actionable."** This is the hardest part for AI -- translating findings into organizational implications. We tuned the prompt to make this specific: not "this is important" but "this supports expanding our CHW program" or "this is too preliminary to act on."

4. **"Your analyst's time is freed up for higher-value work."** Instead of spending 2 hours writing bullet points, she can spend 15 minutes reviewing AI-generated digests and adding organizational context that only she knows. Her expertise shifts from writing to curating.

---

# Scenario 4: CONTENT GENERATION -- Personalized Donor Communications

---

## Customer Brief

**Who they are**: A mid-size nonprofit with a fundraising team of 4 people. They send approximately 500 thank-you letters per month.

**What they said**:

> "We have a generic template but it feels impersonal. Our donors can tell it's a form letter. We want each letter to feel personal based on who the donor is -- how much they gave, how long they've been giving, which programs they support. Our major donors ($5,000+) especially deserve something special. But we can't write 500 unique letters a month."

**Constraints**:
- Must match the org's brand voice: warm, professional, not overly formal, not preachy
- Must accurately reference the donor's giving history
- Major donors ($5,000+) should get longer, more detailed letters
- Must avoid making up impact claims -- only reference real programs

## Discovery Notes

1. **"Can you share your current template and a few letters your best fundraiser has written?"** -- This gives us the voice and tone to match. Key pattern: they always lead with gratitude, mention the specific program, connect to a real outcome, and close with a forward-looking statement.

2. **"What data do you have on each donor?"** -- Name, gift amount, gift date, lifetime giving total, first gift date, years as donor, program(s) supported, any notes from their fundraiser.

3. **"What are the donor tiers and how should they differ?"** -- First-time donors: welcoming, explain impact. Recurring (2+ years): acknowledge loyalty. Major ($5K+): more detail, mention specific projects, personal tone from ED.

4. **"What should we absolutely NOT do?"** -- Never mention other donors' gifts. Never imply their gift is small. Never make up statistics. Never use guilt-based language.

## Solution

In [ ]:
DONOR_LETTER_SYSTEM = """You are a communications writer for a nonprofit organization called "Bridges to Opportunity" that runs education and workforce development programs.

<role>
Write personalized thank-you letters to donors based on their giving history and profile. Each letter should feel genuinely personal -- not like a template with mail-merge fields swapped in.
</role>

<brand_voice>
- WARM but professional. Like a letter from a trusted friend who happens to run a nonprofit.
- SPECIFIC, not generic. Reference the donor's actual contribution and the program it supports.
- GRATEFUL without being obsequious. One "thank you" is enough -- don't grovel.
- FORWARD-LOOKING. End with what's next, not just what happened.
- CONCISE. Respect the donor's time. First-time and recurring donor letters: 150-200 words. Major donor letters: 250-350 words.
- NEVER use guilt, pressure, or imply the donor should give more.
- NEVER fabricate statistics or impact numbers. Only reference what's in the provided program info.
</brand_voice>

<programs>
Bridges to Opportunity runs three programs:
1. "ReadyWork" -- Job training and placement for adults without college degrees. Places 200+ adults per year. 78% job retention rate at 12 months.
2. "YouthLaunch" -- After-school mentoring and college prep for high school students. Serves 350 students. 92% graduate high school, 68% enroll in college.
3. "FreshStart" -- Financial literacy and emergency assistance for families in crisis. Serves 400 families. Average family becomes financially stable within 8 months.
</programs>

<donor_tiers>
Adapt your letter based on the donor's tier:

FIRST_TIME (first gift ever):
- Welcome them to the community
- Explain the specific impact of their gift on the program they supported
- Keep it short and warm
- Sign from "The Bridges to Opportunity Team"

RECURRING (2+ years of giving):
- Acknowledge their loyalty and cumulative impact
- Reference how long they've been giving
- Share a forward-looking update on the program
- Sign from "Maria Torres, Executive Director"

MAJOR ($5,000+ gift):
- Most personal and detailed
- Reference their total giving history
- Connect their gift to specific, tangible outcomes
- Mention an upcoming opportunity (event, site visit, advisory role)
- Sign from "Maria Torres, Executive Director" with a handwritten P.S.
</donor_tiers>

<output_format>
Write ONLY the letter text. Start with the salutation ("Dear [Name],") and end with the signature. No preamble or explanation.
</output_format>

<examples>
<example>
<donor_data>
Name: Patricia Owens
Gift amount: $100
Gift date: November 15, 2024
Tier: FIRST_TIME
Program supported: ReadyWork
Notes: Found us through a friend's Facebook post
</donor_data>
<letter>
Dear Patricia,

Thank you for your generous $100 gift to our ReadyWork program. Welcome to the Bridges to Opportunity family.

Your contribution goes directly to job training for adults who are building new careers without a college degree. Last year, ReadyWork placed over 200 people in jobs -- and 78% of them were still employed a year later. Your gift helps make that possible.

We're so glad your friend shared our work. If you'd like to see it firsthand, we host open house tours on the first Friday of every month. We'd love to show you around.

With gratitude,
The Bridges to Opportunity Team
</letter>
</example>
</examples>"""

# Test with a first-time donor
first_time_donor = """Name: David Kim
Gift amount: $75
Gift date: January 8, 2025
Tier: FIRST_TIME
Program supported: YouthLaunch
Notes: Donated through our year-end email campaign"""

result = call_claude(DONOR_LETTER_SYSTEM, f"<donor_data>\n{first_time_donor}\n</donor_data>")
print_result("First-Time Donor Letter", result)

## Quick Eval

In [ ]:
donor_profiles = [
    {
        "label": "First-Time Donor ($75)",
        "data": """Name: David Kim
Gift amount: $75
Gift date: January 8, 2025
Tier: FIRST_TIME
Program supported: YouthLaunch
Notes: Donated through our year-end email campaign""",
    },
    {
        "label": "Loyal Recurring Donor (7 years, $250)",
        "data": """Name: Margaret and Robert Sullivan
Gift amount: $250
Gift date: December 20, 2024
Tier: RECURRING
Years giving: 7
Lifetime total: $1,850
Program supported: FreshStart
Notes: Retired school teachers. Increase their gift by $25 each year. Attended our gala in 2022.""",
    },
    {
        "label": "Major Donor ($10,000)",
        "data": """Name: Angela Reeves
Gift amount: $10,000
Gift date: November 30, 2024
Tier: MAJOR
Years giving: 4
Lifetime total: $32,500
Program supported: ReadyWork
Notes: CEO of a local staffing firm. Has hired 3 ReadyWork graduates. Interested in expanding employer partnerships. Attended last two galas as a table sponsor.""",
    },
]

print("SCENARIO 4: Donor Communications -- Tier Comparison")
print("=" * 70)

for profile in donor_profiles:
    result = call_claude(DONOR_LETTER_SYSTEM, f"<donor_data>\n{profile['data']}\n</donor_data>")
    print_result(profile["label"], result)

## Demo Talking Points

1. **"Each letter feels genuinely different."** Compare the three letters side by side. David gets a warm welcome. The Sullivans get acknowledged for 7 years of growing support. Angela gets a detailed letter from the ED that references her company's direct involvement. These don't feel like templates.

2. **"The tone adapts, not just the content."** First-time donors get a simple, friendly tone. Longtime donors get a more familiar, collegial tone. Major donors get a more substantive, partnership-oriented tone. This mirrors what your best fundraiser does naturally.

3. **"It uses real program data."** Every impact number in these letters comes from the program descriptions you gave us. Nothing is fabricated. When the YouthLaunch letter mentions 92% graduation rates, that's your real number.

4. **"The workflow is simple."** You export your donor list from your CRM with the fields we specified. The system generates all 500 letters. Your team reviews and sends. Total time: 30 minutes of review instead of days of writing.

---

# Scenario 5: Q&A OVER DOCUMENTS -- Program FAQ Assistant

---

## Customer Brief

**Who they are**: A social services organization that runs multiple programs (food assistance, job training, childcare subsidies). They have a 50-page program guide that staff and clients constantly reference.

**What they said**:

> "Our front desk staff gets the same 20 questions every day. 'Am I eligible for food assistance?' 'How do I apply for childcare subsidies?' 'What documents do I need?' The answers are all in our program guide but nobody reads it. We want a chatbot that can answer these questions accurately. But it absolutely CANNOT make things up -- if someone gets wrong eligibility information, that's a real problem."

**Constraints**:
- Must ONLY answer from the provided document
- Must say "I don't have that information" rather than hallucinate
- Clients may ask in informal language or broken English
- Answers affect people's access to critical services -- accuracy is paramount

## Discovery Notes

1. **"What does your program guide cover?"** -- Eligibility criteria, application processes, required documents, program descriptions, office hours, and contact info for each of 4 programs.

2. **"What are the most dangerous errors?"** -- Telling someone they're eligible when they're not (wastes their time and the org's). Telling someone they're NOT eligible when they are (they miss out on services). Getting document requirements wrong (they show up unprepared).

3. **"What questions does the guide NOT answer?"** -- Individual case status, wait times (they change), anything about programs from other organizations. These are the hallucination traps.

4. **"Who are the users?"** -- Both clients (public-facing) and front desk staff (internal). Clients may have limited English, low literacy, or be stressed and confused.

## Solution

In [ ]:
# Simulated program guide excerpt (in production this would be retrieved via RAG)
PROGRAM_GUIDE = """COMMUNITY SERVICES PROGRAM GUIDE -- Updated January 2025

====================================================================
PROGRAM 1: COMMUNITY FOOD ASSISTANCE (CFA)
====================================================================

Description: The Community Food Assistance program provides monthly food boxes and 
grocery store vouchers to eligible households. Each household receives one food box 
(approximately $150 value) and $75 in grocery vouchers per month.

Eligibility:
- Household income at or below 185% of the Federal Poverty Level (FPL)
- Must reside in Jefferson County
- All household members must be documented (valid ID required for head of household)
- No asset test
- Households currently receiving SNAP benefits ARE eligible (benefits stack)

Income Limits (2025, 185% FPL):
- 1 person: $27,861/year
- 2 persons: $37,814/year
- 3 persons: $47,767/year
- 4 persons: $57,720/year
- Each additional person: add $9,953

Required Documents:
- Government-issued photo ID for head of household
- Proof of Jefferson County residency (utility bill, lease, or official mail dated within 90 days)
- Proof of income for all household members (pay stubs from last 30 days, benefit award letters, or tax return)
- Self-declaration form (provided at intake) if income is from informal employment

Application Process:
1. Walk in to any Community Services office during intake hours (Mon-Thu, 9am-3pm)
2. Complete the CFA application form (available in English, Spanish, and Vietnamese)
3. Provide required documents
4. Staff will verify eligibility within 5 business days
5. If approved, first food box is available for pickup within 10 business days

Renewal: Must recertify every 6 months. Sinply bring updated income documentation.

====================================================================
PROGRAM 2: CHILDCARE SUBSIDY PROGRAM (CSP)
====================================================================

Description: The Childcare Subsidy Program helps eligible families pay for licensed 
childcare. The subsidy covers up to 80% of childcare costs at participating providers. 
Families pay a copay based on a sliding scale.

Eligibility:
- Household income at or below 250% of FPL
- At least one parent/guardian must be working, in school, or in a job training program
  (minimum 20 hours per week)
- Child must be under age 13 (or under 19 if the child has a documented disability)
- Must reside in Jefferson County
- Must use a state-licensed childcare provider (list available at our offices)

Copay Schedule (monthly, per child):
- Income 0-100% FPL: $0
- Income 101-150% FPL: $50
- Income 151-200% FPL: $100
- Income 201-250% FPL: $175

Required Documents:
- Government-issued photo ID for parent/guardian
- Birth certificate or proof of age for each child
- Proof of Jefferson County residency
- Proof of income (last 30 days)
- Proof of work, school enrollment, or job training participation
  (employer letter, school schedule, or training program enrollment letter)
- Name and license number of chosen childcare provider

Application Process:
1. Schedule an appointment by calling (555) 234-5678
2. Attend intake appointment with all required documents
3. Staff will process application within 10 business days
4. If approved, subsidy begins on the 1st of the following month

IMPORTANT: Childcare subsidies CANNOT be applied retroactively. The subsidy starts from 
the date of approval, not the date of application.

====================================================================
PROGRAM 3: JOB READINESS TRAINING (JRT)
====================================================================

Description: A free 8-week job training program covering resume writing, interview 
skills, computer basics, and workplace communication. Graduates receive job placement 
assistance for 6 months after completion.

Eligibility:
- Must be 18 years or older
- Must reside in Jefferson County
- Must be currently unemployed or underemployed (working fewer than 20 hours/week)
- No income requirements
- Must be able to attend classes in person (no virtual option currently available)

Schedule: Classes held Tuesday and Thursday, 6pm-9pm, at the Downtown Community Center
(100 Main Street, Suite 200)

Next Session Start Dates:
- February 4, 2025
- April 15, 2025
- July 8, 2025
- October 7, 2025

How to Enroll:
1. Attend an orientation session (held the Wednesday before each session start date, 6pm)
2. Bring a valid photo ID
3. No application fee

====================================================================
GENERAL INFORMATION
====================================================================

Office Locations:
- Downtown: 100 Main Street, Suite 100 (Mon-Fri, 8am-5pm)
- Eastside: 4500 Oak Avenue (Mon-Wed, 9am-4pm)
- Northgate: 200 Pine Road (Thu-Fri, 9am-4pm)

Phone: (555) 234-5678 (Mon-Fri, 8am-5pm)
Website: www.communityservices-jc.org
Languages: English, Spanish, Vietnamese interpreters available at all locations

Emergency Assistance: For immediate food needs, call 211 or visit the Downtown office 
walk-in hours (Mon-Thu, 9am-3pm). Same-day emergency food boxes are available while 
supplies last (no eligibility screening required for emergency boxes)."""


FAQ_SYSTEM = f"""You are a helpful assistant for Community Services of Jefferson County. You answer questions from clients and staff about our programs and services.

<role>
Answer questions ONLY based on the information in the program guide provided below. You are a trusted source of information that affects people's access to critical services. Accuracy is more important than helpfulness.
</role>

<program_guide>
{PROGRAM_GUIDE}
</program_guide>

<rules>
1. ONLY answer based on information explicitly stated in the program guide above. Do not use outside knowledge.
2. If the answer is NOT in the program guide, say: "I don't have that information in our program guide. Please call us at (555) 234-5678 or visit one of our offices for help with this question."
3. If a question is partially answerable (some info is in the guide, some isn't), answer what you can and clearly state what you cannot answer.
4. When discussing eligibility, be precise. Quote the exact criteria. Do not round numbers or simplify requirements.
5. If someone describes their situation and asks if they're eligible, walk through EACH criterion explicitly. If you cannot determine eligibility from the information they've provided, ask them for the missing details.
6. Use simple, clear language. Many clients have limited English. Avoid jargon.
7. If a question is about a specific program, cite the program name in your answer.
8. NEVER make up program details, eligibility criteria, phone numbers, addresses, or dates that are not in the guide.
</rules>

<tone>
Friendly, patient, and clear. Like a helpful front desk staff member who wants to make sure you understand everything.
</tone>"""

# Quick test
test_q = "I make $45,000 a year and have 3 kids. Can I get help with food?"
result = call_claude(FAQ_SYSTEM, test_q)
print_result("Test: Eligibility Question", result)

## Quick Eval

In [ ]:
faq_test_cases = [
    {
        "label": "Answerable: Specific eligibility question",
        "question": "I'm a single mom making $30,000 a year with two kids. My daughter is 4 and my son is 10. Can I get help paying for daycare?",
        "expected_behavior": "Should walk through CSP eligibility criteria, check income against FPL for household of 3, confirm children's ages qualify, ask about work/school status.",
    },
    {
        "label": "Answerable: Document requirements",
        "question": "What do I need to bring to apply for food help?",
        "expected_behavior": "Should list CFA required documents clearly.",
    },
    {
        "label": "NOT in document: Wait time question",
        "question": "How long is the waiting list for the childcare subsidy right now?",
        "expected_behavior": "Should say 'I don't have that information' -- wait times are not in the guide.",
    },
    {
        "label": "NOT in document: Other org's program",
        "question": "Can you help me apply for Section 8 housing vouchers?",
        "expected_behavior": "Should say this is not a program we offer / not in the guide. Should NOT make up information about Section 8.",
    },
    {
        "label": "Edge case: Informal language",
        "question": "yo i need a job bad. im 22 and aint working rn. yall got something for that?",
        "expected_behavior": "Should recognize this as a JRT inquiry, respond respectfully, and provide enrollment info.",
    },
]

print("SCENARIO 5: FAQ Assistant -- Grounding & Hallucination Tests")
print("=" * 70)

for i, case in enumerate(faq_test_cases, 1):
    result = call_claude(FAQ_SYSTEM, case["question"])
    print(f"\n{'='*70}")
    print(f"  Test {i}: {case['label']}")
    print(f"  Question: {case['question']}")
    print(f"  Expected: {case['expected_behavior']}")
    print(f"{'='*70}")
    print(result)
    print()

## Demo Talking Points

1. **"It answers accurately from your program guide."** When someone asks about food assistance eligibility, it cites the exact income limits from your guide -- $47,767 for a family of 3, not a rounded number. When it lists required documents, it lists exactly what your guide says.

2. **"It refuses to make things up."** Watch what happens when we ask about wait times (not in the guide) or another organization's program (Section 8). Instead of guessing, it says 'I don't have that information' and directs them to call your office. This is the single most important behavior.

3. **"It handles real-world language."** Your clients don't ask textbook questions. They say 'yo i need a job' or 'can I get help with food' in broken English. The system understands the intent and maps it to the right program.

4. **"In production, we'd use RAG."** Right now we're putting the full guide in the prompt. For your 50-page guide, we'd use retrieval-augmented generation to pull in only the relevant sections. This keeps costs down and accuracy up. The prompting pattern stays the same -- we just change how the context gets loaded.

---

# Scenario 6: MULTI-STEP WORKFLOW -- Intake Processing Pipeline

---

## Customer Brief

**Who they are**: A community services organization that receives intake forms from people requesting help. The forms are free-text (people describe their situation in their own words).

**What they said**:

> "When someone fills out our online intake form, we need to: read it, figure out what they need, decide how urgent it is, route it to the right team, and send them an acknowledgment email so they know we received it. Right now that's 4 different steps done by 2 different people and it takes 30-45 minutes per form. We get 50 forms a day."

**Constraints**:
- Each step must be auditable (they need to see what happened at each stage)
- Urgency classification must be conservative (flag high when in doubt)
- The acknowledgment email must reference the specific services mentioned

## Discovery Notes

1. **"Walk me through the current process step by step."** -- (1) Staff reads the form and highlights key info. (2) They classify urgency as Critical/High/Standard. (3) They assign it to one of 4 teams: Housing, Benefits, Employment, or Crisis. (4) They draft a quick email to the person confirming receipt and next steps.

2. **"What makes something Critical vs. High vs. Standard?"** -- Critical: immediate danger (homelessness tonight, no food, domestic violence, mental health crisis). High: urgent but not immediate (eviction in 2 weeks, benefits expiring, job loss). Standard: non-urgent requests (job training interest, general info).

3. **"What goes in the acknowledgment email?"** -- Confirms receipt, mentions what services they'll be connected to, gives a timeline for follow-up (Critical: same day, High: 1-2 business days, Standard: 3-5 business days), and provides the crisis hotline number.

4. **"Why not do this in a single prompt?"** -- We want each step to be a focused task with clear input/output. This makes it easier to debug, audit, and improve individual steps. If extraction is wrong, we fix the extraction prompt without touching the rest.

## Solution

In [ ]:
# ================================================================
# STEP 1: EXTRACT key information from intake form
# ================================================================

STEP1_EXTRACT_SYSTEM = """You are an intake form processor. Extract structured information from a free-text intake form submission.

<task>
Read the intake form and extract the following fields. If a field is not mentioned or unclear, mark it as "NOT_PROVIDED".
</task>

<output_format>
Return ONLY valid JSON with these fields:
{
  "client_name": "string",
  "contact_phone": "string",
  "contact_email": "string",
  "household_size": "integer or NOT_PROVIDED",
  "primary_need": "string -- one sentence describing the main thing they need help with",
  "secondary_needs": ["list of other needs mentioned"],
  "current_situation": "string -- brief summary of their circumstances",
  "time_sensitivity": "string -- any deadlines or urgent dates mentioned",
  "has_children": "boolean or NOT_PROVIDED",
  "employment_status": "string or NOT_PROVIDED",
  "housing_status": "string or NOT_PROVIDED"
}
</output_format>"""

# ================================================================
# STEP 2: CLASSIFY urgency level
# ================================================================

STEP2_URGENCY_SYSTEM = """You are an urgency classifier for a community services intake system.

<task>
Given the extracted information from an intake form, classify the urgency level.
</task>

<urgency_levels>
CRITICAL -- Immediate danger or crisis. Examples:
- Homeless tonight or within 48 hours
- No food and no resources to get food today
- Domestic violence or safety threat
- Mental health crisis or suicidal ideation
- Children in unsafe conditions
Timeline: Must be contacted SAME DAY.

HIGH -- Urgent but not immediately dangerous. Examples:
- Eviction notice with court date in next 30 days
- Benefits expiring or recently terminated
- Just lost job, bills due soon
- Utilities shutoff scheduled
Timeline: Must be contacted within 1-2 BUSINESS DAYS.

STANDARD -- Important but not time-sensitive. Examples:
- Interested in job training programs
- General questions about available services
- Looking for childcare options (not immediate)
- Wants to apply for programs proactively
Timeline: Contacted within 3-5 BUSINESS DAYS.
</urgency_levels>

<rules>
- When in doubt between two levels, ALWAYS choose the MORE urgent level.
- If children are involved AND there's housing or food insecurity, automatically classify as at least HIGH.
- Any mention of safety concerns = CRITICAL, regardless of how casually it's mentioned.
</rules>

<output_format>
Return ONLY valid JSON:
{
  "urgency": "CRITICAL or HIGH or STANDARD",
  "reasoning": "2-3 sentences explaining why this level was chosen",
  "escalation_flags": ["list of any specific concerns that should be flagged for supervisors"]
}
</output_format>"""

# ================================================================
# STEP 3: ROUTE to correct department
# ================================================================

STEP3_ROUTING_SYSTEM = """You are a routing assistant for a community services intake system.

<task>
Given the extracted information and urgency classification, assign this case to the correct team.
</task>

<teams>
CRISIS_TEAM -- Handles all CRITICAL urgency cases regardless of primary need. Also handles domestic violence, mental health crises, and immediate safety concerns.

HOUSING_TEAM -- Eviction prevention, rental assistance, shelter placement, utility assistance, landlord disputes.

BENEFITS_TEAM -- Government benefits enrollment (SNAP, Medicaid, SSI), benefits appeals, food assistance programs.

EMPLOYMENT_TEAM -- Job training, resume help, job placement, career counseling, childcare for working parents.
</teams>

<routing_rules>
- ALL cases classified as CRITICAL urgency go to CRISIS_TEAM, regardless of the primary need.
- For HIGH and STANDARD cases, route based on the primary need.
- If the primary need is ambiguous, route to the team that handles the most urgent sub-issue.
- Childcare for working parents routes to EMPLOYMENT_TEAM (not BENEFITS_TEAM).
</routing_rules>

<output_format>
Return ONLY valid JSON:
{
  "assigned_team": "CRISIS_TEAM or HOUSING_TEAM or BENEFITS_TEAM or EMPLOYMENT_TEAM",
  "reasoning": "1-2 sentences",
  "secondary_referrals": ["list of other teams that should be notified about secondary needs"]
}
</output_format>"""

# ================================================================
# STEP 4: DRAFT acknowledgment email
# ================================================================

STEP4_EMAIL_SYSTEM = """You are a communications assistant drafting acknowledgment emails for a community services organization.

<task>
Draft a brief, compassionate acknowledgment email to the client confirming we received their intake form and telling them what to expect next.
</task>

<email_guidelines>
- Start with "Dear [Name]" (use their name from the extracted info)
- Confirm we received their submission
- Briefly acknowledge their situation (show we read their form) WITHOUT repeating sensitive details
- Mention the specific services we'll be connecting them to
- Give the response timeline based on urgency:
  - CRITICAL: "We will contact you TODAY"
  - HIGH: "We will contact you within 1-2 business days"
  - STANDARD: "We will contact you within 3-5 business days"
- Always include: "If you are in immediate danger, please call 911. For crisis support, call our 24-hour hotline at (555) 999-HELP."
- Sign from "Community Services Intake Team"
- Tone: warm, professional, reassuring. These people are often stressed and scared.
- Keep it under 150 words.
</email_guidelines>

<output_format>
Return ONLY the email text. No preamble.
</output_format>"""

print("Pipeline prompts defined. Ready to test.")

In [ ]:
def run_intake_pipeline(intake_text, verbose=True):
    """Run the full 4-step intake processing pipeline."""
    results = {}

    # Step 1: Extract
    if verbose:
        print("\n" + "=" * 70)
        print("  STEP 1: EXTRACTION")
        print("=" * 70)
    extracted_raw = call_claude(STEP1_EXTRACT_SYSTEM, f"<intake_form>{intake_text}</intake_form>")
    try:
        extracted = json.loads(extracted_raw)
    except json.JSONDecodeError:
        extracted = {"raw": extracted_raw, "error": "Failed to parse JSON"}
    results["extraction"] = extracted
    if verbose:
        print(json.dumps(extracted, indent=2))

    # Step 2: Classify urgency (input: extraction output)
    if verbose:
        print("\n" + "=" * 70)
        print("  STEP 2: URGENCY CLASSIFICATION")
        print("=" * 70)
    urgency_raw = call_claude(
        STEP2_URGENCY_SYSTEM,
        f"<extracted_info>{json.dumps(extracted, indent=2)}</extracted_info>"
    )
    try:
        urgency = json.loads(urgency_raw)
    except json.JSONDecodeError:
        urgency = {"raw": urgency_raw, "error": "Failed to parse JSON"}
    results["urgency"] = urgency
    if verbose:
        print(json.dumps(urgency, indent=2))

    # Step 3: Route (input: extraction + urgency)
    if verbose:
        print("\n" + "=" * 70)
        print("  STEP 3: ROUTING")
        print("=" * 70)
    routing_input = f"""<extracted_info>{json.dumps(extracted, indent=2)}</extracted_info>
<urgency_classification>{json.dumps(urgency, indent=2)}</urgency_classification>"""
    routing_raw = call_claude(STEP3_ROUTING_SYSTEM, routing_input)
    try:
        routing = json.loads(routing_raw)
    except json.JSONDecodeError:
        routing = {"raw": routing_raw, "error": "Failed to parse JSON"}
    results["routing"] = routing
    if verbose:
        print(json.dumps(routing, indent=2))

    # Step 4: Draft email (input: extraction + urgency + routing)
    if verbose:
        print("\n" + "=" * 70)
        print("  STEP 4: ACKNOWLEDGMENT EMAIL")
        print("=" * 70)
    email_input = f"""<extracted_info>{json.dumps(extracted, indent=2)}</extracted_info>
<urgency_classification>{json.dumps(urgency, indent=2)}</urgency_classification>
<routing>{json.dumps(routing, indent=2)}</routing>"""
    email_text = call_claude(STEP4_EMAIL_SYSTEM, email_input)
    results["email"] = email_text
    if verbose:
        print(email_text)

    return results

print("Pipeline function defined.")

## Quick Eval

In [ ]:
# Test Case 1: CRITICAL -- family in crisis
intake_form_1 = """Name: Rosa Gutierrez
Phone: 555-0192
Email: rosa.g@email.com

Please describe your situation:
My husband left three weeks ago and I don't have enough money for rent. The landlord 
gave me a notice that I have until Friday to pay or he's changing the locks. I have 
three kids (ages 2, 5, and 8) and my mom lives with us too -- she's 72 and has diabetes. 
I work part-time at a grocery store but it's only about 15 hours a week and I make $12/hr.
I don't know what to do. We might be on the street this weekend. I also ran out of my 
mom's diabetes medication and can't afford to refill it. I feel like I'm falling apart. 
Is there any help at all? I'll take anything."""

print("SCENARIO 6: Intake Pipeline")
print("=" * 70)
print("TEST CASE 1: Family in Crisis")
print(f"\nINPUT:\n{intake_form_1}")
results_1 = run_intake_pipeline(intake_form_1)

In [ ]:
# Test Case 2: STANDARD -- job training inquiry
intake_form_2 = """Name: Marcus Williams
Phone: 555-0384

Please describe your situation:
Hi, I recently finished my GED and I'm looking to get into a better career. I've been 
doing warehouse work for the past 5 years but I want to learn computer skills and maybe 
get an office job. I heard you guys have some kind of job training program? I'm 
available evenings and weekends. I make about $35,000 a year right now and I live alone. 
No rush on this -- just want to start planning for the future. Thanks!"""

print("\n\n" + "#" * 70)
print("TEST CASE 2: Job Training Inquiry")
print(f"\nINPUT:\n{intake_form_2}")
results_2 = run_intake_pipeline(intake_form_2)

## Demo Talking Points

1. **"This is prompt chaining, not a single monolithic prompt."** Four focused prompts, each doing one job well. The extraction prompt doesn't need to know about urgency rules. The urgency prompt doesn't need to know how to write emails. This is modular -- you can improve any step independently.

2. **"Every step is auditable."** Look at the pipeline output: you can see exactly what was extracted, why the urgency was classified at that level, why it was routed to that team, and what email was drafted. If something is wrong, you can pinpoint which step failed and fix it.

3. **"The safety net works."** Rosa's case was classified CRITICAL because of imminent homelessness with children and the emotional distress signals. Even though her stated need was rent help (Housing), the urgency override sent it to the Crisis Team. The email promises same-day contact and includes the crisis hotline.

4. **"Marcus's case is handled efficiently."** A standard job training inquiry gets routed to the Employment Team with a 3-5 day response timeline. No over-escalation, no wasted crisis resources.

5. **"Cost and latency."** Four API calls per form, each focused and fast. At roughly $0.01-0.03 per call with Claude Sonnet, that's about $0.05-0.12 per intake form. At 50 forms per day, that's $2.50-6.00/day vs. 2 staff members spending 30-45 minutes each on manual processing.

---

# Evaluation Patterns: Reusable Quick-Eval Framework

---

Use this pattern in any scenario to quickly test and demo your solution.

In [ ]:
def quick_eval(system_prompt, test_cases, model=MODEL, max_tokens=2048):
    """Run test cases against a system prompt and display results in a clean format.

    Args:
        system_prompt: The system prompt to test.
        test_cases: List of dicts, each with:
            - 'label': Short description of the test case
            - 'input': The user message to send
            - 'expected': (optional) What you expect to see in the output
            - 'check_fn': (optional) A function that takes the output and returns (bool, str)
        model: Model ID to use.
        max_tokens: Max tokens for each response.

    Returns:
        List of result dicts with 'label', 'input', 'output', 'expected', and 'passed' fields.
    """
    results = []
    total = len(test_cases)

    print(f"\nRunning {total} test cases...")
    print("=" * 80)

    for i, case in enumerate(test_cases, 1):
        label = case["label"]
        user_input = case["input"]
        expected = case.get("expected", None)
        check_fn = case.get("check_fn", None)

        # Call Claude
        try:
            output = call_claude(system_prompt, user_input, model=model, max_tokens=max_tokens)
        except Exception as e:
            output = f"ERROR: {e}"

        # Run check function if provided
        passed = None
        check_msg = ""
        if check_fn:
            try:
                passed, check_msg = check_fn(output)
            except Exception as e:
                passed = False
                check_msg = f"Check function error: {e}"

        # Display result
        status = ""
        if passed is True:
            status = "[PASS]"
        elif passed is False:
            status = "[FAIL]"
        else:
            status = "[----]"

        print(f"\n{status} Test {i}/{total}: {label}")
        print(f"  Input: {user_input[:100]}{'...' if len(user_input) > 100 else ''}")
        if expected:
            print(f"  Expected: {expected}")
        if check_msg:
            print(f"  Check: {check_msg}")
        print(f"  Output:")
        # Indent output for readability
        for line in output.split("\n"):
            print(f"    {line}")

        results.append({
            "label": label,
            "input": user_input,
            "output": output,
            "expected": expected,
            "passed": passed,
        })

    # Summary
    print("\n" + "=" * 80)
    checked = [r for r in results if r["passed"] is not None]
    if checked:
        passed_count = sum(1 for r in checked if r["passed"])
        print(f"Results: {passed_count}/{len(checked)} checks passed")
    else:
        print(f"Results: {total} cases run (no automated checks)")
    print("=" * 80)

    return results


print("quick_eval() defined and ready to use.")

In [ ]:
# Demo: Using quick_eval with automated checks

def check_has_category(output):
    """Check that the output contains a valid category tag."""
    valid_categories = ["MENTAL_HEALTH", "LEGAL_AID", "BENEFITS_ENROLLMENT",
                        "HOUSING_ASSISTANCE", "GENERAL_INQUIRY"]
    for cat in valid_categories:
        if cat in output:
            return True, f"Found category: {cat}"
    return False, "No valid category found in output"


def check_category_is(expected_category):
    """Return a check function that verifies a specific category."""
    def checker(output):
        if f"<category>{expected_category}</category>" in output:
            return True, f"Correct: {expected_category}"
        # Also check without XML tags
        if expected_category in output:
            return True, f"Category {expected_category} found (may not be in XML tags)"
        return False, f"Expected {expected_category}, not found in output"
    return checker


def check_valid_json(output):
    """Check that the output is valid JSON."""
    try:
        json.loads(output)
        return True, "Valid JSON"
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}"


# Run quick_eval on the ticket routing scenario
routing_eval_cases = [
    {
        "label": "Clear benefits case",
        "input": "<email>My SNAP benefits were cut off and I need help getting them reinstated. Case number BEN-44921.</email>",
        "expected": "BENEFITS_ENROLLMENT",
        "check_fn": check_category_is("BENEFITS_ENROLLMENT"),
    },
    {
        "label": "Crisis detection",
        "input": "<email>I just need some information about your housing programs. Things have been really hard and honestly I've been thinking about ending it all. But anyway do you have any apartments available?</email>",
        "expected": "MENTAL_HEALTH (crisis signal: 'ending it all')",
        "check_fn": check_category_is("MENTAL_HEALTH"),
    },
    {
        "label": "Legal matter",
        "input": "<email>My employer hasn't paid me in three weeks. When I asked about it they threatened to report me to immigration. I have a valid work permit. What are my rights?</email>",
        "expected": "LEGAL_AID",
        "check_fn": check_category_is("LEGAL_AID"),
    },
]

results = quick_eval(TICKET_ROUTING_SYSTEM, routing_eval_cases)

In [ ]:
# Demo: Using quick_eval for the extraction scenario with JSON validation

extraction_eval_cases = [
    {
        "label": "Complete application",
        "input": f"<application>{sample_application_1}</application>",
        "expected": "Valid JSON with all fields populated",
        "check_fn": check_valid_json,
    },
    {
        "label": "Incomplete application (should have flags)",
        "input": f"<application>{sample_application_3}</application>",
        "expected": "Valid JSON with MISSING/UNCLEAR flags",
        "check_fn": check_valid_json,
    },
]

results = quick_eval(GRANT_EXTRACTION_SYSTEM, extraction_eval_cases)

---

# Summary: Patterns to Internalize

| Pattern | Key Prompt Techniques | Common Pitfalls |
|---------|----------------------|------------------|
| **Classification** | Few-shot examples with reasoning, priority rules, confidence scores | Categories that overlap without clear rules |
| **Extraction** | Strict output schema, MISSING/UNCLEAR flags, no-guess rules | Model inventing data for missing fields |
| **Summarization** | Audience definition, evidence quality rubric, output template | Too technical for audience, burying the "so what" |
| **Content Generation** | Brand voice guidelines, tier-based adaptation, few-shot examples | Generic sounding output, fabricated statistics |
| **Q&A over Docs** | Grounding instructions, explicit refusal behavior, context in prompt | Hallucinating answers not in the document |
| **Multi-step Pipeline** | Prompt chaining, focused single-task prompts, structured handoffs | Steps that are too coupled, error propagation |

## The Meta-Pattern for Any Customer Scenario

1. **Listen and discover.** Ask what they actually need (not what they think they need). Get examples.
2. **Define the input/output contract.** What goes in? What comes out? What format?
3. **Write the system prompt.** Role, rules, examples, output format -- in that order.
4. **Test on diverse inputs.** Include happy path, edge cases, adversarial cases, and empty/malformed input.
5. **Show the customer.** Demo with their data. Explain what it does well, where it needs human review, and how it integrates.

---

**Next steps**: Practice running these scenarios end-to-end. Time yourself. In the interview, you'll have about 15-20 minutes to go from customer brief to working demo. These patterns should be muscle memory.